# Imports

In [2]:
import chromadb
import json
import logging
import os
import uuid
import xml.etree.ElementTree as ET
import yaml

from bs4 import BeautifulSoup
from chromadb import Search, K, Knn
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from pathlib import Path

# Definitions and Credentials
- Define functions that are relevant for several cells. 
- Load credentials for model interaction.

In [3]:
def get_config():
    with open("./RAG_Test/configs/config.yaml", "r") as fp:
        return yaml.safe_load(fp)

def get_metadataProject():
    
    path = Path(config["paths"]["metadata"])

    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w") as f:
            json.dump({}, f)
    
    with open(path, "r") as f:
        return json.load(f)
    
def initialze_chrobadb(config, metadataProject):
    chroma_client = chromadb.PersistentClient(path=config['paths']['dataBase_dir'])
    collection = chroma_client.get_or_create_collection(
        name=config['paths']['dataBase_name'],
        embedding_function=None,
        configuration={
            "hnsw": {
                "space": "cosine"
                }
            }
        )
    
    print(f"Starting chromadb \"{config['paths']['dataBase_name']}\"\n")
    return chroma_client, collection
    
def refresh_db(collection):
        all_ids = collection.get(include=[])["ids"]
        if all_ids:
            collection.delete(ids=all_ids)

#Get credenitals
config = get_config()
load_dotenv(config['paths']['env_dir'])
client = OpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("API_KEY"),
)

# Collect Source Files
- Search all directories / subdirectories for files with file extensions defined in the config file.
- Generate a uuid for each file and return a .json-like entry.
- Save directories, filenames and type of file in metadataProject.json with uuid as primary key. 

In [ ]:
%%time

def find_documents(extensions, metadataProject, root):
        
    root_path = Path(root)
    
    for file in root_path.rglob("*"): 
        for extension in extensions:
           if file.is_file() and file.suffix.lower() == extension:
            metadataProject[str(uuid.uuid4())] = {
                'directory': str(file.parent.resolve()),
                'fileName': str(file.name),
                'type': extension 
            }
            
    return metadataProject

#Start
config = get_config()
metadataProject = {}

extensions = config['settings']['file_extensions']
root = config['paths']['data_raw']

#Search for matching files formats and return metadata in a .json object
metadataProject = find_documents(extensions, metadataProject, root)

#Save project metadata for the next step
with open(config['paths']['metadata'], "w") as f:
    json.dump(metadataProject, f, indent=4)

#create a log message
sliceNumber = 5
print(f"Chosen file extensions: {extensions}\n")
print(f"In total {len(metadataProject)} files were found and registred. Here the first {sliceNumber} elements:\n")

for x, obj in list(metadataProject.items())[:sliceNumber]:
    print(obj['fileName'])
#print time
print()
result = sum(range(10_000_000))

# Create Chunks
- Parse jats .xml files and extract title and doi.
- Tie up metadata package according to chromadb schema.
- Split texts in chunks.
- Save chunks together with metadata in chromadb.
- For traceability purposes, also save the chunks together with the metadata as a text file in the "processed" folder.

Important:
- In line with the concept of this notebook to use a sequence of cells, the embeddings are not yet generated at this stage. However, as chromadb expects embeddings, they are simulated here in the dimension corresponding to the model. These simulated embeddings are then replaced in the next step.
- The functions for parsing .md and .txt files are placeholder functions.

In [ ]:
%%time

def add_packages_to_chromadb(articleSets, config, collection):

    for articleSet in articleSets:
      for i, (uuid, articlePackage), in enumerate(articleSet.items(), start=1):
        if i % config['settings']['log_ratio_file'] == 0: 
            print(f"Following file was splitted: {uuid}, with the titel: \"{articlePackage['metadatas'][0]['article_title']}\".")
            print(f"In total a number of {len(articlePackage['ids'])} chunks were recorded.\n")
        
        collection.upsert(
          ids = articlePackage['ids'],
          documents = articlePackage['documents'],
          embeddings = articlePackage['embeddings'],
          metadatas = articlePackage['metadatas'],
        )  

def create_articlePackages(config, metadataProject, splitter):
  articleSet = {}
  articleSets = []

  print(f"Starting...\n\n{len(metadataProject)} file(s) to process.\n\nCaution: Log message only every {config['settings']['log_ratio_file']}th file. \n")
    
  for i, (uuid, fileInfo) in enumerate(metadataProject.items(), start=1):
    if i % config['settings']['log_ratio_file'] == 0:
        print(f"File {i} / {len(metadataProject)}, file {uuid}")
    
    #Initialize empty article package according to chromadb metadata schema
    articlePackage = {}
    articlePackage['ids'] = []
    articlePackage['documents'] = []
    articlePackage['metadatas'] = []
    articlePackage['embeddings'] = []
    
    #Other
    all_sections = []
    metadataArticle = {}
    metadataArticle['ID'] = uuid
    
    toOpen = os.path.join(fileInfo['directory'], fileInfo['fileName'])
    fileObj = Path(toOpen)
    
    #Extract all texts and (if available) metadata  
    if fileObj.suffix.lower() == '.xml':
      metadataArticle, all_sections = parse_jatsXML(toOpen, metadataArticle, all_sections) 
    
    if fileObj.suffix.lower() == '.txt':
      metadataArticle, all_sections = parse_plain_txt(toOpen, metadataArticle, all_sections)
      
    if fileObj.suffix.lower() == '.md':
        metadataArticle, all_sections = parse_markdown_docs(toOpen, metadataArticle, all_sections)
      
    #Split texts, prepare metadata for chromadb metadata schema and create one data package for each article
    articlePackage, chunkNumber = split_text_in_sections(splitter, metadataArticle, all_sections, articlePackage, config)
         
    #Save full package for each article
    pathToSave = os.path.join(config['paths']['data_chunks'], uuid)
    
    fileInfo['number_of_chunks'] = chunkNumber # needed for the entry in metadataProject
    with open(pathToSave, 'w') as fp:
      json.dump(articlePackage , fp)
     
    #Save project metadata for the next step
    with open(config['paths']['metadata'], "w") as f:
      json.dump(metadataProject, f, indent=4)

    articleSet[uuid] = articlePackage
  
  articleSets.append(articleSet)

  return articleSets

def create_embedding(chunk):
  """ 
  CAUTION: This function creates a dummy embedding with the dimensions
  of the vector the selected model will produce.
  The idea is to run the scripts separately in the jupyter notebook,
  i. e. the embeddings will be created in the next step.
  """
  
  embedding = [0.0] * config['settings']['dimension_embedding_vector']
  
  return embedding

def extract_sections(sec, parent_titles=None):
    if parent_titles is None:
        parent_titles = []

    chunks = []

    title_tag = sec.find("title", recursive=False)
    title = title_tag.get_text(" ", strip=True) if title_tag else "Untitled"

    current_titles = parent_titles + [title]

    #Collect direct paragraphs only
    paragraphs = sec.find_all("p", recursive=False)

    text = "\n".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
    )

    if text.strip():
        chunks.append({
            "section_path": " > ".join(current_titles),
            "text": text
        })

    #Recurse into subsections
    child_secs = sec.find_all("sec", recursive=False)

    for child in child_secs:
        chunks.extend(extract_sections(child, current_titles))

    return chunks

def get_text_or_none(node, *args, **kwargs):
    element = node.find(*args, **kwargs)

    if element:
        return element.get_text(" ", strip=True)
    else:
        return None
    
    #return element.get_text(" ", strip=True) if element else None

def parse_jatsXML(toOpen, metadataArticle, all_sections):
  with open(toOpen, 'r', encoding="utf-8") as file:
    soup = BeautifulSoup(file, features="xml")

    metadataArticle['title'] = get_text_or_none(soup, "article-title")
    metadataArticle['doi'] = get_text_or_none(
        soup,
        "article-id",
        {"pub-id-type": "doi"}
    )
    
    #Remove paragraph numbers
    paragraphNumbers = soup.find_all('named-content')
    for paragraphNumber in paragraphNumbers:
      paragraphNumber.decompose()

    #Now get the text from the article sections containing <P> text
    sections = soup.find_all("sec")
    for sec in sections:
      all_sections.extend(extract_sections(sec))

    """
    In case there are no sections containing <p> content:
    get all the <p> content and add NA section informations
    """
    if len(all_sections) == 0:
      text = "\n".join(
        p.get_text(strip=True)
        for p in soup.find_all("p")
      )
      section =  {}
      section['name'] = 'NA'
      section['section_path'] = 'NA'
      section['text'] = text
    
      all_sections.append(section)

    return metadataArticle, all_sections

def parse_markdown_docs(toOpen, metadataArticle, all_sections):
    """
    This function depends on the structure of the original document.
    Some source file may contain more structured information.
    In this case the function needs to be adjusted.
    """

    from langchain_text_splitters import MarkdownHeaderTextSplitter

    with open(toOpen, 'r', encoding="utf-8") as file:
          sourceTextLines = file.readlines()
        
    sourceText = ' '.join(sourceTextLines)

    header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")]
    )

    sections = header_splitter.split_text(sourceText)
    
    metadataArticle['title'] = Path(toOpen).stem #Filename as alternative for missing title
    metadataArticle['doi'] = 'NA'
    metadataArticle["section_path"] = 'NA'
    section ={}
    section['name'] = 'NA'
    section['section_path'] = 'NA'
  
    section['text'] = sections[0].page_content
   
    all_sections.append(section)
   
    return metadataArticle, all_sections

def parse_plain_txt(toOpen, metadataArticle, all_sections):
      """
      In case of plain text the needed set of metadata is put together manually
      """
      
      with open(toOpen, 'r', encoding="utf-8") as file:
          sourceTextLines = file.readlines()
    
      sourceText = ' '.join(sourceTextLines)
    
      metadataArticle['title'] = Path(toOpen).stem #Filename as alternative for missing title
      metadataArticle['doi'] = 'NA'
      metadataArticle["section_path"] = 'NA'
      section ={}
      section['name'] = 'NA'
      section['section_path'] = 'NA'
      
      section['text'] = sourceText
    
      all_sections.append(section)
    
      return metadataArticle, all_sections

def split_text_in_sections(splitter, metadataArticle, all_sections, articlePackage, config):
  chunks = []
  counter = 0
  
  for section in all_sections:
      split_texts = splitter.split_text(section["text"])
      articlePackage, counter = tie_up_package(metadataArticle, section, split_texts, articlePackage, counter, config)
  
  return articlePackage, counter

def tie_up_package(metadataArticle, section, split_texts, articlePackage, counter, config):
  for i, chunk in enumerate(split_texts):
          counter += 1
          embedding = create_embedding(chunk)
          articlePackage['ids'].append(metadataArticle['ID'] + '_' + str(counter))
          articlePackage['documents'].append(chunk)
          articlePackage['embeddings'].append(embedding),
          metadata = {
                  "article_title": metadataArticle['title'],
                  "article_uuid": metadataArticle['ID'],
                  "doi": metadataArticle['doi'],
                  "section": section["section_path"],
                  "chunk_ID": i
          }
          articlePackage['metadatas'].append(metadata)
          
  return articlePackage, counter

#Start
config = get_config()
metadataProject = get_metadataProject()
chroma_client, collection = initialze_chrobadb(config, metadataProject)

#Initialize the splitter
splitter = RecursiveCharacterTextSplitter(chunk_size = config['settings']['chunk_size'], chunk_overlap = config['settings']['chunk_overlap'])

#Create a chunk package for every article
articleSets = create_articlePackages(config, metadataProject, splitter)

#Add entries to chromadb and display log
add_packages_to_chromadb(articleSets, config, collection)

print(f"Finished. Status: {collection.count()} entries in {config['paths']['dataBase_name']}")

#print time
print()
result = sum(range(10_000_000))

# Create Embeddings
Create embeddings with the selected model and replace dummy embeddings from the step above for each entry in chromadb.

In [ ]:
%%time

def create_embeddings_chunk(config, chunk):
    try:
        response = client.embeddings.create(
            model=config['settings']['embedding-model'],
            input=chunk
        )
        embedding_obj = response.data[0]
        vector = embedding_obj.embedding

        return vector
        
    except Exception as e:
        print(type(e))
        print(e)

def create_embeddings_collection(config, collection):
    all_ids = collection.get()["ids"]
    print(f"{len(all_ids)} embeddings need to be created ...\n")
    print(f"Starting. Caution: Log message every {config['settings']['log_ratio_chunk']}th action. \n")
    for i, id in enumerate(all_ids, start=1):
        result = collection.get(
            ids=id,
            include=["documents"]
        )
        new_embedding = create_embeddings_chunk(config, result['documents'][0])
        if i % config['settings']['log_ratio_chunk'] == 0:
            print(f"Created a new embedding ({i} / {len(all_ids)}) for {id} (starting with {new_embedding[:3]})")
        
        collection.update(
            ids=[id],
            embeddings=[new_embedding]
        )

#Start
config = get_config()
metadataProject = get_metadataProject()
chroma_client, collection = initialze_chrobadb(config, metadataProject)

create_embeddings_collection(config, collection)
print("\n")
print(f"Finished. Status: {collection.count()} entries in {config['paths']['dataBase_name']}")

#print time
print()
result = sum(range(10_000_000))

### Retrieval and Content Generation
- Get question, query chromadb and create an answer.

In [ ]:
%%time

def create_query_embedding(question):
    #Create query embedding
    query_embedding = client.embeddings.create(
                model=config['settings']['embedding-model'],
                input=question
            )
    
    embedding_obj = query_embedding.data[0]
    vector = embedding_obj.embedding
    
    #Retrieve documents
    results = collection.query(
        query_embeddings=[vector],
        n_results=5,
        #Optional: For filtering out weak matches before sending them to the llm:
        include=["documents", "distances"]
    )
    
    docs = []
    for doc, distance in zip(
            results["documents"][0],
            results["distances"][0]):
        if distance < 0.5:  #Threshold can be adjusted
            docs.append(doc)
    
    context = "\n\n".join(docs)
       
    return context

def create_answer(question, context):

    prompt = f"""
    Use the following context to answer the question. Be concise and clear.
    
    Context:
    {context}
    
    Question:
    {question}
    """
    
    response = client.chat.completions.create(
        model=config['settings']['generation-model'],
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. "
                    "Answer if possible from the provided context. "
                    "If the answer is not in the context, say so."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
            
        ]
    )
    
    answer = response.choices[0].message.content
    
    return answer

#Start
config = get_config()
metadataProject = get_metadataProject()
chroma_client, collection = initialze_chrobadb(config, metadataProject)

question = input("Enter your question here: ")
print("\n [... working ...]\n")
context = create_query_embedding(question)

answer = create_answer (question, context)
print(f"Answer: {answer}")

#print time
print()
result = sum(range(10_000_000))

# Appendix: Predefined Scripts for interacting with the db
This collection of predefined scripts provides quickly an illustrative and useful insight into the database and shows how to interact with chromadb. 

In [ ]:
#Start
config = get_config()
metadataProject = get_metadataProject()
chroma_client, collection = initialze_chrobadb(config, metadataProject)

print(f"Following databases are found: {chroma_client.list_collections()}\n")
print(f"Acitve database: {config['paths']['dataBase_name']}, containing {collection.count()} entries.\n")

#Show embedding vector dimension
run = False
if run:
    result = collection.get(
                include=["embeddings"], 
                limit=5
    )
    print(f"Embedding vector dimension in {config['paths']['dataBase_name']}: {len(result['embeddings'][0])}\n")

#Show metadata schema
run = True
if run:
    print("Example standard metadata schema in chromadb:\n")
    results = collection.get()
    for x, y in results.items():
        print(f"{x} | {type(y)}")
    print("\n---------------\n\nmetadatas used here:")
    for x, y in results["metadatas"][0].items():
        print(f"{x}: {y}")

#Show example metadata
run = True
if run:
    print("\nExample ids and metadata:\n")
    results = collection.get(
        include=["metadatas"],
        limit=10
    )
    for id_, metadata in zip(results["ids"], results["metadatas"]):
        print(f"ID: {id_}")
        print(f"Metadata: {metadata}")
        print()

#Show keys and metadata example
run = True
if run:
    results = collection.get()
    print(results.keys())
    print(results["ids"][:5])
    print(results["metadatas"][:5])

#Show chunks
run = True
if run:
    print("\nExample chunks:\n")
    result = collection.get(
                include=["documents"], 
                limit=5
    )
    for i, entry in enumerate(result['documents']):
        print(f"Chunk {i}: {entry}\n")

#Show embeddings
run = True
if run:
    print("\nExample Embeddings:\n")
    result = collection.get(
                include=["embeddings"], 
                #limit=5
    )
    for i, entry in enumerate(result['embeddings'][:5]):
        print(f"Chunk {i}: {entry}\n")

#Search by specific id
run = False
if run:
    results = collection.get(ids=['########################'], include=['embeddings'])
    print(results)

#Show uuids
run = False
if run:
    results = collection.get(
    include=["metadatas"]
    )

    uuids = [
        metadata["article_uuid"]
        for metadata in results["metadatas"]
        if metadata and "article_uuid" in metadata
    ]

    for id in set(uuids):
        print(id)


# Appendix: Clean up Project Folder and refresh chromadb
Deleting files in the "processed" folder that were created in previous runs and that are not longer in use (meaning: every file that is not listed in the current metadataProcet.json file)

In [ ]:
#Start
config = get_config()
metadataProject = get_metadataProject()
chroma_client, collection = initialze_chrobadb(config, metadataProject)

#Get uuids that are currently in use
currentFiles = []

for key, value in metadataProject.items():
    currentFiles.append(key)
    
#Collect all file names in "processed" folder
directory = Path(config['paths']['data_chunks'])
dirContent = directory.iterdir()

allFiles = []
for file in dirContent:
    if file.is_file():
        allFiles.append(file.name)

#Get difference
toDelete = set(allFiles) - set(currentFiles)

if len(toDelete) == 0:
    print(f"Nothing to clean up")
else: 
    print(f"{len(toDelete)} files to delete.\n")

number = []
for filename in toDelete:
    file_path = os.path.join(config['paths']['data_chunks'], filename)
    fileObj = Path(file_path)

    if fileObj.exists() and fileObj.is_file():
        fileObj.unlink()
        number.append(filename)
    else:
        print(f"Not found: {fileObj}")
print(f"{len(number)} files deleted")
print(number[:5])
#Delete all entries in chromadb (CAUTION!)
run = True
if run:
    refresh_db(collection)
    print(f"All entries in {config['paths']['dataBase_name']} deleted")
    print(f"Acitve database: {config['paths']['dataBase_name']}, containing {collection.count()} entries.\n")